# Project 3: update after feedback

This supplement follows the submitted Project 3 notebook. It checks how the NVE regions are combined, shows how to request a larger electricity-area sample, and tests the analysis settings. The submitted notebook and screencast remain available from the Project 3 overview.

## Combined NVE weather regions

I combine the source polygons before sampling: **East = RS + RØ**, **West = RV**, and **North = RM + RN**. I draw 1,000 reproducible, area-weighted points from each combined region. These NVE administrative regions are an approximation to precipitation regions, not exact catchment boundaries. The electricity zones NO1–NO5 are a different set of polygons, so I keep their samples separate.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Find the repository root whether the notebook is opened here or from VS Code.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "streamlit_app.py").exists())
sys.path.insert(0, str(root))
from utils.quality_utils import (build_combined_weather_sample, build_electricity_sample,
    load_nve_regions, load_electricity_areas, load_open_meteo_region,
    spc_precipitation_plot, lof_temperature_plot)
regions = load_nve_regions()
weather_points = build_combined_weather_sample(regions, n_each=1000)
electricity_points = build_electricity_sample(load_electricity_areas(), n_each=1000)
display(weather_points.groupby("region_group").size().rename("locations per combined region"))
display(electricity_points.groupby("area_group").size().rename("locations per electricity area"))

2026-11-16 12:03:42.968 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-16 12:03:42.969 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-16 12:03:42.969 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-16 12:03:42.970 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


region_group
East     1000
North    1000
West     1000
Name: locations per combined region, dtype: int64

area_group
NO1    1000
NO2    1000
NO3    1000
NO4    1000
NO5    1000
Name: locations per electricity area, dtype: int64

In [2]:
# Check that each coordinate is actually inside the intended combined polygon.
groups = {"East": ["RS", "RØ"], "West": ["RV"], "North": ["RM", "RN"]}
for name, codes in groups.items():
    selected = weather_points[weather_points.region_group == name]
    points = gpd.GeoSeries(gpd.points_from_xy(selected.longitude, selected.latitude), crs="EPSG:4326")
    polygon = regions[regions.reg_code.isin(codes)].geometry.union_all()
    assert len(points) == 1000 and points.covered_by(polygon).all()
print("All 1,000 points are inside each combined region.")

All 1,000 points are inside each combined region.


## Weather download and the sample size

The original 2024 NO3 example used five points from 1,000 generated points. That is a small sample and it does not represent all local conditions. The downloader now accepts up to all 1,000 coordinates. It requests batches and checks that every requested point and hour is present before calculating the mean. In the app, five points remain the quick default; 1,000 can be selected explicitly.

Open-Meteo limits free requests and counts longer periods and multiple locations towards its quota. A full year for 1,000 points may exceed those limits. An initial one-day request for 1,000 points reached the rate limit. On 16 November 2026 I retried with smaller batches spaced over time and successfully downloaded 24 hourly averages for all 1,000 NO3 points on 1 January 2024. The following saved file contains these averages, not the original location-by-location responses. A full year for all 1,000 points **has not been downloaded or analysed**. Completed batch summaries are cached for up to seven days, including on local disk when writable, so a request can be retried later. Cloud redeployment can clear this cache. The original five-point 2024 results remain in the submitted notebook.

To retry all 1,000 points for NO3, use the call below in a new code cell when the API quota permits. A successful result is an hourly mean across all requested points; no partial average is accepted.

```python
weather = load_open_meteo_region(
    "NO3", "2024-01-01", "2024-12-31",
    point_count=1000, sample_points=electricity_points
)
```

For a combined NVE weather region, the same function accepts `"East"`, `"West"` or `"North"` and uses that region’s own 1,000-point sample. This is separate from the NO1–NO5 electricity-area sample used by the app.

## Real NO3 example: 1,000 locations for one day

This saved result is from the actual Open-Meteo ERA5 request described above. The two measurements are hourly means across all 1,000 sampled NO3 coordinates, not the full-year analysis. No simulated data are used in this cell.

In [3]:
# Read the saved aggregate from the successful 1,000-point request.
one_day = pd.read_json(root / "projects/project-3/updates/NO3_1000_points_2024-01-01.json")
one_day["time"] = pd.to_datetime(one_day["time"])
assert len(one_day) == 24 and one_day[["precipitation", "temperature_2m"]].notna().all().all()
print("Actual Open-Meteo result: 1,000 locations, 24 hourly averages for NO3 on 1 January 2024.")
display(one_day.head())

Actual Open-Meteo result: 1,000 locations, 24 hourly averages for NO3 on 1 January 2024.


,time,precipitation,temperature_2m
0,2024-01-01 00:00:00,0.0046,-3.4794
1,2024-01-01 01:00:00,0.0042,-3.5544
2,2024-01-01 02:00:00,0.0035,-3.7023
3,2024-01-01 03:00:00,0.0033,-3.6899
4,2024-01-01 04:00:00,0.0033,-3.7306


## Checking parameter choices

The next test uses **synthetic data**, not the NO3 weather download. It checks that changing the SPC limit and the LOF anomaly proportion changes the summaries. The 1% LOF default comes from the task; it is a chosen proportion, not a probability that an observation is wrong.

In [4]:
rng = np.random.default_rng(320)
t = np.arange(240)
test_weather = pd.DataFrame({
    "time": pd.date_range("2024-01-01", periods=len(t), freq="h"),
    "precipitation": np.maximum(0, rng.gamma(1.3, 0.3, len(t))),
    "temperature_2m": 3 + 4 * np.sin(2 * np.pi * t / 24) + rng.normal(0, 0.6, len(t)),
})
test_weather.loc[20, "precipitation"] = 6
test_weather.loc[60, "temperature_2m"] = 15
results = []
for multiplier in (2.0, 3.0):
    figure, summary = spc_precipitation_plot(test_weather, cutoff=0.08, n_sd=multiplier)
    plt.close(figure)
    results.append({"method": "SPC", "setting": f"{multiplier:g} SD", "marked": summary["outliers"]})
for proportion in (0.01, 0.02):
    figure, summary = lof_temperature_plot(test_weather, contamination=proportion)
    plt.close(figure)
    results.append({"method": "LOF", "setting": f"{proportion:.0%}", "marked": summary["anomalies"]})
comparison = pd.DataFrame(results)
display(comparison)
assert comparison.loc[1, "marked"] <= comparison.loc[0, "marked"]
assert comparison.loc[3, "marked"] >= comparison.loc[2, "marked"]

,method,setting,marked
0,SPC,2 SD,25
1,SPC,3 SD,8
2,LOF,1%,3
3,LOF,2%,5


The weekly reservoir series uses an STL period of 52 because 52 weeks are approximately one year. It is an approximation: some calendar years contain an extra ISO week.

## AI use for this update

AI assistance was used to implement and test this feedback update. The original notebook describes AI use for the submitted work.